In [ ]:
import kagglehub

path = kagglehub.dataset_download("phucthaiv02/butterfly-image-classification")
print("Path to dataset files:", path)

In [ ]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
        break

In [ ]:
import pandas as pd
from PIL import Image
import numpy as np
import os


train_csv = os.path.join(path, "Training_set.csv")
test_csv  = os.path.join(path, "Testing_set.csv")
train_dir = os.path.join(path, "train")
test_dir  = os.path.join(path, "test")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
from PIL import Image


train_csv = os.path.join(path, "Training_set.csv")
train_df = pd.read_csv(train_csv)
train_dir = os.path.join(path, "train")

rows = []
cols = []
labels = []

for _, row in train_df.iterrows():
    img_path = os.path.join(train_dir, row['filename'])
    img = Image.open(img_path)
    w, h = img.size  
    cols.append(w)
    rows.append(h)
    labels.append(row['label'])

rows = np.array(rows)
cols = np.array(cols)
labels = np.array(labels)

plt.figure(figsize=(8,6))
plt.scatter(cols, rows, alpha=0.5)
plt.xlabel('Width (pixels)')
plt.ylabel('Height (pixels)')
plt.title('Scatter of Image Sizes (Width x Height)')
plt.show()

class_counts = pd.Series(labels).value_counts().sort_values(ascending=False)

plt.figure(figsize=(12,6))
class_counts.plot(kind='bar')
plt.xlabel('Class')
plt.ylabel('Number of images')
plt.title('Number of images per class')
plt.xticks(rotation=45, ha='right')
plt.show()

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg


img_path = os.path.join(path, 'train', 'Image_1.jpg')


img = mpimg.imread(img_path)
plt.imshow(img)
plt.axis('off')
plt.show()

In [ ]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.ToTensor(),  
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])  
])

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd
import os

train_csv = os.path.join(path, "Training_set.csv")
train_df = pd.read_csv(train_csv)


train_df, val_df = train_test_split(train_df, test_size=0.2, stratify=train_df['label'], random_state=42)

print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Count images per class
train_counts = train_df['label'].value_counts().sort_index()
val_counts   = val_df['label'].value_counts().sort_index()

print("=== Training set counts per class ===")
print(train_counts)
print("\n=== Validation set counts per class ===")
print(val_counts)

In [ ]:
plt.figure(figsize=(14,6))

# Training set
plt.subplot(1,2,1)
train_counts.plot(kind='bar', color='skyblue')
plt.title("Number of images per class (Training Set)")
plt.xlabel("Class")
plt.ylabel("Number of images")
plt.xticks(rotation=45, ha='right')

# Validation set
plt.subplot(1,2,2)
val_counts.plot(kind='bar', color='salmon')
plt.title("Number of images per class (Validation Set)")
plt.xlabel("Class")
plt.ylabel("Number of images")
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
#CNN
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, losses
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import os
import gc


IMG_HEIGHT = 128      
IMG_WIDTH = 128
BATCH_SIZE = 32
NUM_EPOCHS = 15

csv_path = "/kaggle/input/datasets/phucthaiv02/butterfly-image-classification/Training_set.csv"
image_folder = "/kaggle/input/datasets/phucthaiv02/butterfly-image-classification/train"

df = pd.read_csv(csv_path)


label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['label'])
num_classes = len(label_encoder.classes_)

from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(
    df, test_size=0.2, stratify=df['label'], random_state=42
)

del df
gc.collect()

def process_image(file_path, label, augment=False):
    img = tf.io.read_file(file_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_HEIGHT, IMG_WIDTH])
    img = img / 255.0

    if augment:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_brightness(img, 0.1)
        img = tf.image.random_contrast(img, 0.9, 1.1)
    return img, label

def df_to_dataset(df, shuffle=True, batch_size=BATCH_SIZE, augment=False):
    file_paths = df['filename'].apply(lambda x: os.path.join(image_folder, x)).values
    labels = df['label'].values

    ds = tf.data.Dataset.from_tensor_slices((file_paths, labels))
    ds = ds.map(lambda x, y: process_image(x, y, augment), num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(df), 1000))  
    ds = ds.batch(batch_size, drop_remainder=True)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = df_to_dataset(train_df, shuffle=True, augment=True)
val_ds = df_to_dataset(val_df, shuffle=False, augment=False)

def build_cnn(num_classes, input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)):
    model = models.Sequential([
        layers.Input(shape=input_shape),

        layers.Conv2D(32, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Conv2D(64, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Conv2D(128, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.GlobalAveragePooling2D(),  
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

model = build_cnn(num_classes)

model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss=losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

model.summary()


history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=NUM_EPOCHS
)

In [ ]:
import matplotlib.pyplot as plt
def history_plot(hist,name):
   
    acc = hist.history['accuracy']
    val_acc = hist.history['val_accuracy']
    loss = hist.history['loss']
    val_loss = hist.history['val_loss']
    
    epochs_range = range(len(acc))
    
    plt.figure(figsize=(14,5))
    
    # Accuracy
    plt.subplot(1,2,1)
    plt.plot(epochs_range, acc, label='Training Accuracy', marker='o')
    plt.plot(epochs_range, val_acc, label='Validation Accuracy', marker='o')
    plt.title(name+': Training and Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    
    # Loss
    plt.subplot(1,2,2)
    plt.plot(epochs_range, loss, label='Training Loss', marker='o')
    plt.plot(epochs_range, val_loss, label='Validation Loss', marker='o')
    plt.title(name+': Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()

In [ ]:
import matplotlib.pyplot as plt

acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(len(acc))

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1,2,2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_true = []
y_pred = []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)


cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=label_encoder.classes_)

plt.figure(figsize=(12,12))
disp.plot(cmap=plt.cm.Blues, values_format='d')
plt.title("Confusion Matrix")
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

y_true = []
y_pred = []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

cm = confusion_matrix(y_true, y_pred)

cm_df = pd.DataFrame(cm, index=label_encoder.classes_, columns=label_encoder.classes_)

print("Confusion Matrix (Counts):")
print(cm_df)

In [ ]:
correct = (y_true == y_pred)

true_count = np.sum(correct)
false_count = len(correct) - true_count

print(f"Total True (correct predictions): {true_count}")
print(f"Total False (wrong predictions): {false_count}")

In [ ]:
import pandas as pd

results_df = pd.DataFrame({
    'True Label': [label_encoder.classes_[i] for i in y_true],
    'Correct': correct
})

summary = results_df.groupby('True Label')['Correct'].value_counts().unstack(fill_value=0)
summary = summary.rename(columns={True: 'True', False: 'False'})

print("True/False count per class:")
print(summary)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def model_inference_visualizer(model, dataset, label_encoder, num_images=25):
    
    images_shown = 0
    plt.figure(figsize=(15, 15))

    for images, labels in dataset:
        preds = model.predict(images, verbose=0)
        pred_classes = np.argmax(preds, axis=1)
        for i in range(len(images)):
            if images_shown >= num_images:
                break

            true_label = label_encoder.classes_[labels[i].numpy()]
            pred_label = label_encoder.classes_[pred_classes[i]]
            correct = (true_label == pred_label)

            plt.subplot(5, 5, images_shown + 1)
            plt.imshow(images[i].numpy())
            plt.axis('off')
            title_color = 'green' if correct else 'red'
            plt.title(f"T:{true_label}\nP:{pred_label}", color=title_color, fontsize=10)

            images_shown += 1

        if images_shown >= num_images:
            break

    plt.tight_layout()
    plt.show()


model_inference_visualizer(model, val_ds, label_encoder, num_images=25)

In [ ]:
def occlusion_sensitivity(model, image, class_idx, patch_size=32):
    h, w, c = image.shape
    heatmap = np.zeros((h, w))
    for i in range(0, h, patch_size):
        for j in range(0, w, patch_size):
            img_copy = image.copy()
            img_copy[i:i+patch_size, j:j+patch_size, :] = 0
            pred = model.predict(img_copy[None, ...])
            heatmap[i:i+patch_size, j:j+patch_size] = pred[0, class_idx]
    return heatmap

# Usage
for images, labels in val_ds.take(1):
    img = images[0].numpy()
    class_idx = np.argmax(model.predict(img[None,...])[0])
    heatmap = occlusion_sensitivity(model, img, class_idx)
    plt.imshow(img)
    plt.imshow(heatmap, cmap='jet', alpha=0.5)
    plt.show()

In [ ]:
img_tensor = tf.convert_to_tensor(img[None,...])
with tf.GradientTape() as tape:
    tape.watch(img_tensor)
    pred = model(img_tensor)
    class_idx = tf.argmax(pred[0])
    loss = pred[:, class_idx]

grads = tape.gradient(loss, img_tensor)
saliency = tf.reduce_max(tf.abs(grads), axis=-1)[0]
plt.imshow(img)
plt.imshow(saliency, cmap='hot', alpha=0.5)
plt.show()

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import math

def compute_saliency(model, img, class_idx=None):
    img_tensor = tf.convert_to_tensor(img[None, ...])
    img_tensor = tf.cast(img_tensor, tf.float32)

    with tf.GradientTape() as tape:
        tape.watch(img_tensor)
        preds = model(img_tensor, training=False)

        if class_idx is None:
            class_idx = tf.argmax(preds[0])

        loss = preds[:, class_idx]

    grads = tape.gradient(loss, img_tensor)
    saliency = tf.reduce_max(tf.abs(grads), axis=-1)[0]

    # Normalize
    saliency = (saliency - tf.reduce_min(saliency)) / \
               (tf.reduce_max(saliency) - tf.reduce_min(saliency) + 1e-8)

    return saliency.numpy()

def occlusion_sensitivity(model, img, class_idx=None, patch_size=20):
    img = img.numpy() if isinstance(img, tf.Tensor) else img
    img_copy = img.copy()
    h, w, c = img_copy.shape

    heatmap = np.zeros((h, w))

    # Get predicted class
    if class_idx is None:
        preds = model.predict(img_copy[None, ...], verbose=0)
        class_idx = np.argmax(preds[0])

    baseline_pred = model.predict(img_copy[None, ...], verbose=0)[0, class_idx]

    for i in range(0, h, patch_size):
        for j in range(0, w, patch_size):
            img_temp = img_copy.copy()
            img_temp[i:i+patch_size, j:j+patch_size, :] = 0

            pred = model.predict(img_temp[None, ...], verbose=0)[0, class_idx]
            heatmap[i:i+patch_size, j:j+patch_size] = baseline_pred - pred

    # Normalize
    heatmap = (heatmap - np.min(heatmap)) / \
              (np.max(heatmap) - np.min(heatmap) + 1e-8)

    return heatmap

num_images = 6   

images_list = []
labels_list = []
preds_list = []

for imgs, labels in val_ds:
    preds = model.predict(imgs, verbose=0)
    pred_classes = np.argmax(preds, axis=1)

    images_list.extend(imgs.numpy())
    labels_list.extend(labels.numpy())
    preds_list.extend(pred_classes)

    if len(images_list) >= num_images:
        break

cols = 3
rows = math.ceil(num_images / cols)

plt.figure(figsize=(15, 5 * rows))

for i in range(num_images):
    img = images_list[i]
    true_label = label_encoder.classes_[labels_list[i]]
    pred_label = label_encoder.classes_[preds_list[i]]
    correct = true_label == pred_label

    saliency = compute_saliency(model, img, preds_list[i])
    overlay = 0.6 * img + 0.4 * np.stack([saliency]*3, axis=-1)

    plt.subplot(rows, cols, i + 1)
    plt.imshow(overlay)
    plt.title(f"Saliency\nT:{true_label} | P:{pred_label}",
              color='green' if correct else 'red')
    plt.axis('off')

plt.tight_layout()
plt.show()

plt.figure(figsize=(15, 5 * rows))

for i in range(num_images):
    img = images_list[i]
    true_label = label_encoder.classes_[labels_list[i]]
    pred_label = label_encoder.classes_[preds_list[i]]
    correct = true_label == pred_label

    occlusion = occlusion_sensitivity(
        model,
        img,
        class_idx=preds_list[i],
        patch_size=20
    )

    overlay = 0.6 * img + 0.4 * np.stack([occlusion]*3, axis=-1)

    plt.subplot(rows, cols, i + 1)
    plt.imshow(overlay)
    plt.title(f"Occlusion\nT:{true_label} | P:{pred_label}",
              color='green' if correct else 'red')
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import tensorflow as tf
import gc

tf.keras.backend.clear_session()
gc.collect()

print("CNN Done")

In [ ]:
#VGG16
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import os
import math
import matplotlib.pyplot as plt

BATCH_SIZE = 32
NUM_EPOCHS = 15
IMG_SIZE = 224  

csv_path = "/kaggle/input/datasets/phucthaiv02/butterfly-image-classification/Training_set.csv"
image_folder = "/kaggle/input/datasets/phucthaiv02/butterfly-image-classification/train"

df = pd.read_csv(csv_path)
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['label'])
num_classes = len(label_encoder.classes_)

train_df, val_df = train_test_split(
    df, test_size=0.2, stratify=df['label'], random_state=42
)


In [ ]:
def build_dataset(df, image_folder, img_size, augment=False):
    file_paths = df['filename'].apply(lambda x: os.path.join(image_folder, x)).values
    labels = df['label'].values
    labels = tf.convert_to_tensor(labels, dtype=tf.int32)

    def process(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, [img_size, img_size])
        if augment:
            img = tf.image.random_flip_left_right(img)
            img = tf.image.random_brightness(img, 0.1)
        img = preprocess_input(img)  
        return img, label

    ds = tf.data.Dataset.from_tensor_slices((file_paths, labels))
    ds = ds.map(process, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        ds = ds.shuffle(buffer_size=len(df))
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = build_dataset(train_df, image_folder, IMG_SIZE, augment=True)
val_ds = build_dataset(val_df, image_folder, IMG_SIZE, augment=False)

steps_per_epoch = math.ceil(len(train_df)/BATCH_SIZE)
validation_steps = math.ceil(len(val_df)/BATCH_SIZE)
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, models, optimizers

IMG_HEIGHT = 224
IMG_WIDTH = 224
NUM_CLASSES = 75  
NUM_EPOCHS = 40
BATCH_SIZE = 32


In [ ]:
base_model = VGG16(
    weights='imagenet',      
    include_top=False,       
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)
)


base_model.trainable = False


model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation='softmax')
])


model.compile(
    optimizer=optimizers.Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()



In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15
)

In [ ]:
print(history.history.keys())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('VGG16 Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

plt.figure(figsize=(8,5))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('VGG16 Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_true = []
y_pred = []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=label_encoder.classes_)

plt.figure(figsize=(12,12))
disp.plot(cmap=plt.cm.Blues, values_format='d')
plt.title("Confusion Matrix")
plt.show()

In [ ]:
correct = (y_true == y_pred)

true_count = np.sum(correct)
false_count = len(correct) - true_count

print(f"Total True (correct predictions): {true_count}")
print(f"Total False (wrong predictions): {false_count}")

In [ ]:
import tensorflow as tf
import gc

tf.keras.backend.clear_session()
gc.collect()

print("VGG done")

In [ ]:
#ResNet50
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import ResNet50
import matplotlib.pyplot as plt



AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

IMG_HEIGHT, IMG_WIDTH = 224, 224
BATCH_SIZE = 16
NUM_CLASSES = 75
NUM_EPOCHS_PHASE1 = 5       
NUM_EPOCHS_PHASE2 = 10      
STEPS_PER_EPOCH = 100       
VALIDATION_STEPS = 50


In [ ]:
base_model = ResNet50(
    include_top=False,
    weights='imagenet',
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)
)
base_model.trainable = False   

resnet_model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')
])

resnet_model.compile(
    optimizer=optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

resnet_model.summary()


history_phase1 = resnet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=NUM_EPOCHS_PHASE1,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_steps=VALIDATION_STEPS
)


base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

resnet_model.compile(
    optimizer=optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_phase2 = resnet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=NUM_EPOCHS_PHASE2,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_steps=VALIDATION_STEPS
)


plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history_phase1.history['accuracy'], label='train acc (phase1)')
plt.plot(history_phase1.history['val_accuracy'], label='val acc (phase1)')
plt.plot(history_phase2.history['accuracy'], label='train acc (ft)')
plt.plot(history_phase2.history['val_accuracy'], label='val acc (ft)')
plt.title("ResNet50 Accuracy")
plt.legend()

plt.subplot(1,2,2)
plt.plot(history_phase1.history['loss'], label='train loss (phase1)')
plt.plot(history_phase1.history['val_loss'], label='val loss (phase1)')
plt.plot(history_phase2.history['loss'], label='train loss (ft)')
plt.plot(history_phase2.history['val_loss'], label='val loss (ft)')
plt.title("ResNet50 Loss")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


y_true = []
y_pred = []

for images, labels in val_ds:
    preds = resnet_model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=label_encoder.classes_)


plt.figure(figsize=(12,12))
disp.plot(cmap=plt.cm.Blues, values_format='d')
plt.title("Confusion Matrix")
plt.show()

In [ ]:
correct = (y_true == y_pred)

true_count = np.sum(correct)
false_count = len(correct) - true_count

print(f"Total True (correct predictions): {true_count}")
print(f"Total False (wrong predictions): {false_count}")

In [ ]:
import tensorflow as tf
import gc

tf.keras.backend.clear_session()
gc.collect()

print("ResNet Done")

In [ ]:
#Mobile Net
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import MobileNetV2
import matplotlib.pyplot as plt


IMG_HEIGHT = 224
IMG_WIDTH = 224
NUM_CLASSES = 75   
BATCH_SIZE = 32
NUM_EPOCHS = 20

train_ds = train_ds.map(lambda x, y: (tf.image.resize(x, [IMG_HEIGHT, IMG_WIDTH]), y))
val_ds   = val_ds.map(lambda x, y: (tf.image.resize(x, [IMG_HEIGHT, IMG_WIDTH]), y))

base_model = MobileNetV2(
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  


mobilenet_model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

mobilenet_model.compile(
    optimizer=optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

mobilenet_model.summary()

history_mobilenet = mobilenet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=NUM_EPOCHS,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps
)

base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

mobilenet_model.compile(
    optimizer=optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_finetune = mobilenet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,  
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps
)

plt.figure(figsize=(12,5))

# Accuracy
plt.subplot(1,2,1)
plt.plot(history_mobilenet.history['accuracy'], label='train acc (phase1)')
plt.plot(history_mobilenet.history['val_accuracy'], label='val acc (phase1)')
plt.plot(history_finetune.history['accuracy'], label='train acc (ft)')
plt.plot(history_finetune.history['val_accuracy'], label='val acc (ft)')
plt.title("MobileNetV2 Accuracy")
plt.legend()

# Loss
plt.subplot(1,2,2)
plt.plot(history_mobilenet.history['loss'], label='train loss (phase1)')
plt.plot(history_mobilenet.history['val_loss'], label='val loss (phase1)')
plt.plot(history_finetune.history['loss'], label='train loss (ft)')
plt.plot(history_finetune.history['val_loss'], label='val loss (ft)')
plt.title("MobileNetV2 Loss")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
 
y_true = []
y_pred = []

for images, labels in val_ds:
    preds = mobilenet_model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)


cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=label_encoder.classes_)

plt.figure(figsize=(12,12))
disp.plot(cmap=plt.cm.Blues, values_format='d')
plt.title("Confusion Matrix")
plt.show()

In [ ]:
correct = (y_true == y_pred)

true_count = np.sum(correct)
false_count = len(correct) - true_count

print(f"Total True (correct predictions): {true_count}")
print(f"Total False (wrong predictions): {false_count}")

In [ ]:
import tensorflow as tf
import gc
tf.keras.backend.clear_session()
gc.collect()
print("Mobile net done")


In [ ]:
#Inception
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input
import matplotlib.pyplot as plt

IMG_HEIGHT = 224
IMG_WIDTH = 224
NUM_CLASSES = 75  
BATCH_SIZE = 32
NUM_EPOCHS = 20

base_model = InceptionV3(
    include_top=False,
    weights='imagenet',
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)
)
base_model.trainable = False  

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer=optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=NUM_EPOCHS
)

base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

model.compile(
    optimizer=optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_finetune = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=NUM_EPOCHS
)

plt.figure(figsize=(12,5))

# Accuracy
plt.subplot(1,2,1)
plt.plot(history_phase1.history['accuracy'], label='train acc (phase1)')
plt.plot(history_phase1.history['val_accuracy'], label='val acc (phase1)')
plt.plot(history_finetune.history['accuracy'], label='train acc (ft)')
plt.plot(history_finetune.history['val_accuracy'], label='val acc (ft)')
plt.title("InceptionV3 Accuracy")
plt.legend()

# Loss
plt.subplot(1,2,2)
plt.plot(history_phase1.history['loss'], label='train loss (phase1)')
plt.plot(history_phase1.history['val_loss'], label='val loss (phase1)')
plt.plot(history_finetune.history['loss'], label='train loss (ft)')
plt.plot(history_finetune.history['val_loss'], label='val loss (ft)')
plt.title("InceptionV3 Loss")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_true = []
y_pred = []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)


cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=label_encoder.classes_)


plt.figure(figsize=(12,12))
disp.plot(cmap=plt.cm.Blues, values_format='d')
plt.title("Confusion Matrix")
plt.show()

In [ ]:
correct = (y_true == y_pred)

true_count = np.sum(correct)
false_count = len(correct) - true_count

print(f"Total True (correct predictions): {true_count}")
print(f"Total False (wrong predictions): {false_count}")

In [ ]:
tf.keras.backend.clear_session()
gc.collect()
print("Inception Done")